# PS- 4A: CBOW IMDb Document Classification

This notebook follows the supplied CBOW example style.


In [ ]:
# Import all packages first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [ ]:
# 1. Load updated IMDb data and split it.
df = pd.read_csv("IMDB Dataset.csv").sample(n=10000, random_state=42)
x = df["review"]
y = df["sentiment"].map({"positive": 1, "negative": 0})
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:
# 2. Convert reviews into lists of words.
train_words = [review.lower().split() for review in x_train]
test_words = [review.lower().split() for review in x_test]
print(train_words[0][:20])


In [ ]:
# 3. Train Word2Vec CBOW. sg=0 means CBOW.
cbow = Word2Vec(sentences=train_words, vector_size=100, window=5,
                min_count=2, sg=0, epochs=5)
print("Vocabulary size:", len(cbow.wv))


In [ ]:
# 4. Average word vectors to represent each review.
x_train_embed = np.array([cbow.wv.get_mean_vector(words) for words in train_words])
x_test_embed = np.array([cbow.wv.get_mean_vector(words) for words in test_words])


In [ ]:
# 5. Build and train a feed-forward document classifier.
model = Sequential([
    Dense(64, activation="relu", input_shape=(100,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(x_train_embed, y_train, epochs=5, batch_size=64, validation_split=0.2)


In [ ]:
# 6. Evaluate using loss and accuracy.
loss, accuracy = model.evaluate(x_test_embed, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.legend()
plt.show()
